## Initialisation

Ce notebook a pour objectif d'ingérer les données brutes dans la base PostgreSQL.

On commence par établir la connexion à la base, puis on charge les deux datasets sources :
- **releves_incidents** : journal des incidents machines (opérateur, sévérité, type de défaut, shift…)
- **telemetry** : relevés capteurs par machine (température, pression, tension, rotation, pièces produites)

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text, URL

DB_USER = "indusense_user"
DB_PASSWORD = "ThEP@ssW0rd"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "indusense_db"

url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

engine = create_engine(url)

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print(f"✅ Connexion à PostgreSQL réussie ({DB_HOST}:{DB_PORT}/{DB_NAME})")
except Exception as e:
    print(f"❌ Échec de connexion : {e}")

df_incidents = pd.read_csv("artifacts/ingestions/incidents/releves_incidents.csv")
print(f"✅ releves_incidents.csv chargé : {df_incidents.shape[0]} lignes, {df_incidents.shape[1]} colonnes")

df_telemetry = pd.read_csv("artifacts/ingestions/incidents/telemetry.csv")
print(f"✅ telemetry.csv chargé : {df_telemetry.shape[0]} lignes, {df_telemetry.shape[1]} colonnes")

## Ingestion CSV → PostgreSQL

Insertion des deux datasets dans leurs tables bronze respectives. On utilise `if_exists="replace"` pour recréer les tables à chaque exécution.

In [ ]:
df_incidents["valid_parsing"] = True
df_telemetry["valid_parsing"] = True

try:
    df_incidents.to_sql("bronze_releves_incidents", engine, if_exists="replace", index=False)
    print(f"✅ {len(df_incidents)} lignes insérées dans 'bronze_releves_incidents'")
except Exception as e:
    print(f"❌ Échec pour 'bronze_releves_incidents' : {e}")

try:
    df_telemetry.to_sql("bronze_telemetry", engine, if_exists="replace", index=False)
    print(f"✅ {len(df_telemetry)} lignes insérées dans 'bronze_telemetry'")
except Exception as e:
    print(f"❌ Échec pour 'bronze_telemetry' : {e}")

## Inspection des tables existantes

Avant de créer les modèles SQLAlchemy, on liste toutes les tables présentes dans la base avec leurs colonnes et types.

In [ ]:
from sqlalchemy import inspect

inspector = inspect(engine)

for table_name in inspector.get_table_names():
    print(f"\n📋 {table_name}")
    for col in inspector.get_columns(table_name):
        print(f"   - {col['name']}: {col['type']}")

## Renommage des tables — ajout du préfixe `bronze_`

Les tables `machine` et `maintenance` sont renommées pour suivre la convention de nommage bronze.

In [ ]:
renames = [
    ("machine", "bronze_machine"),
    ("maintenance", "bronze_maintenance"),
]

with engine.begin() as conn:
    for old_name, new_name in renames:
        try:
            conn.execute(text(f'ALTER TABLE "{old_name}" RENAME TO "{new_name}"'))
            print(f"✅ '{old_name}' → '{new_name}'")
        except Exception as e:
            print(f"⚠️  '{old_name}' : {e}")